# Typing, Dataclasses, Enums, singledispatch & Type Introspection

## What you'll practice

Type hints are optional at runtime but powerful for tooling (mypy/pyright, IDEs) — and readable at runtime too, through `dataclasses.fields`, `typing.get_type_hints`,
`get_origin`/`get_args`. You'll build value types with validation, enums, function overloading via `singledispatch`, and a mini 'pydantic' that converts nested dicts into typed dataclasses.
Run `mypy` on your notebook code (export it to a `.py`) once to see what it catches.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/advanced/test_10_02_typing_dataclasses_enums.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/10_advanced_python/02_typing_dataclasses_enums.ipynb`. Don't open it until you've tried.

## Exercise 1 — 🟢 Easy · Enums for statuses

**Implement:** `class OrderStatus(str, Enum)`

Members `PLACED, SHIPPED, DELIVERED, CANCELLED, RETURNED` with values equal to their names. Add `classmethod parse(value)` (case-insensitive, strips; unknown → `ValueError`
listing valid values) and a property `is_terminal` (DELIVERED, CANCELLED, RETURNED). Because it subclasses `str`, members compare equal to plain strings.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_order_status` in `tests/advanced/test_10_02_typing_dataclasses_enums.py`.

**Check:** `pytest tests/advanced/test_10_02_typing_dataclasses_enums.py -k test_order_status`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · A validated, frozen, slotted value type

**Implement:** `class OrderLine`

`@dataclass(frozen=True, slots=True, order=True)` with fields `order_id: str`, `line_no: int`, `product_id: str`, `quantity: int`, `unit_price: Decimal`. `__post_init__` validates
(`line_no >= 1`, `quantity >= 1`, `unit_price >= 0`, ids non-blank) and **coerces** `unit_price` from str/int to `Decimal` (use `object.__setattr__` since it's frozen). Property `amount`.
`from_dict(d)` classmethod. Ordering compares fields in declaration order (so lines sort by order then line number).

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_order_line, test_order_line_validation, test_order_line_float_rejected` in `tests/advanced/test_10_02_typing_dataclasses_enums.py`.

**Check:** `pytest tests/advanced/test_10_02_typing_dataclasses_enums.py -k "test_order_line or test_order_line_validation or test_order_line_float_rejected"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Function overloading with singledispatch

**Implement:** `read_lines(source) -> list[str]`

Java overloads by parameter type; Python uses `functools.singledispatch`. `read_lines` returns stripped, non-blank lines from: a `pathlib.Path` (read the file), a `str`
(**treated as text**, not a path — explicit is better than implicit), `bytes` (UTF-8 decode), any object with a `.read()` method (file-like: `io.StringIO`, open files), or any other
iterable of strings. Register implementations per type.

**Input:** `source`

**Expected output:** `list[str]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_read_lines` in `tests/advanced/test_10_02_typing_dataclasses_enums.py`.

**Check:** `pytest tests/advanced/test_10_02_typing_dataclasses_enums.py -k test_read_lines`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Convert raw rows using a dataclass's type hints

**Implement:** `from_row(cls, row: dict[str, str])`

Generic loader: given any dataclass `cls` and a dict of **strings**, build an instance converting each field according to its annotation (read with `typing.get_type_hints(cls)`).
Support `str`, `int`, `float`, `bool` (true/false/1/0/yes/no), `date`, `datetime` (ISO, `Z` ok), `Decimal`, `Enum` subclasses (by value), and `X | None` / `Optional[X]` (blank → `None`).
Missing non-optional fields without defaults → `TypeError` naming them; blank values for non-optional fields → `ValueError`; fields with defaults may be omitted.

**Input:** `cls, row: dict[str, str]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_from_row, test_from_row_errors` in `tests/advanced/test_10_02_typing_dataclasses_enums.py`.

**Check:** `pytest tests/advanced/test_10_02_typing_dataclasses_enums.py -k "test_from_row or test_from_row_errors"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Typed nested configuration (pydantic-lite) · 🧩 Mini-project

**Implement:** `class ConfigError(ValueError)`, `load_typed(cls, data, path: str = '')`

Convert nested JSON-like data into nested dataclasses, recursively, guided by type hints. Support: dataclasses (dicts → instances; unknown keys → error), `list[X]`, `dict[str, X]`,
`X | None`, `Enum`, `int`/`float`/`str`/`bool` (**no** string coercion here: JSON already has types; but an `int` is acceptable where `float` is expected; `bool` is never an `int`),
and defaults. On any problem raise `ConfigError` with a path like `"jobs[1].retry.max_attempts: expected int, got str"` or `"jobs[0]: unknown keys ['retries']"` or
`"source: missing required field"`. Then define dataclasses for a pipeline config (jobs with sources, retry policy, schedule enum) and load one from JSON.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_load_typed_ok, test_load_typed_errors` in `tests/advanced/test_10_02_typing_dataclasses_enums.py`.

**Check:** `pytest tests/advanced/test_10_02_typing_dataclasses_enums.py -k "test_load_typed_ok or test_load_typed_errors"`

**YOUR CODE ↓**